In [0]:
%sql
-- 1. Create the main Catalog
CREATE CATALOG IF NOT EXISTS ecommerce_ops;
USE CATALOG ecommerce_ops;

-- 2. Create the three Medallion Schemas
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

-- 3. Create a Managed Volume in Bronze for your raw CSV uploads
CREATE VOLUME IF NOT EXISTS ecommerce_ops.bronze.raw_files;


In [0]:
from pyspark.sql.functions import current_timestamp, col

# 1. Read raw CSV from Unity Catalog Volume
raw_path = "/Volumes/ecommerce_ops/bronze/raw_files/"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(raw_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

# 2. Write untouched raw records to Bronze Delta table
(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_ops.bronze.transactions_raw")
)

print("Bronze Ingestion Complete! Row count:", spark.table("ecommerce_ops.bronze.transactions_raw").count())


Bronze Ingestion Complete! Row count: 1000123
